# 02. Distributed Training with Ray Train
© 2026, Anyscale. All Rights Reserved

This notebook takes the single-GPU training loop from notebook 01 and scales it to many GPUs, and many nodes, with Ray Train. It runs a real multi-worker training job, a deliberate worker failure, and a resume, all against `src/train_ray_train.py`, which is the file this notebook teaches.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook </b>

<ol>
  <li>When to use Ray Train</li>
  <li>Update the training loop</li>
  <li>Migrate the model</li>
  <li>Migrate the dataset</li>
  <li>Report metrics and checkpoints</li>
  <li>Set the runtime configuration (RunConfig)</li>
  <li>Configure scale and GPUs (ScalingConfig)</li>
  <li>Launch the distributed training job</li>
  <li>Access the training results</li>
  <li>Use the checkpointed model</li>
  <li>See the workers across nodes</li>
  <li>Recover from a failure, and resume</li>
  <li>Activity: run with four workers</li>
  <li>What changed from v1, and Ray Train in production</li>
</ol>

</div>

**Setup**

In [ ]:
import os
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

import ray
import ray.train
import ray.train.torch
from ray.train import Checkpoint, CheckpointConfig, FailureConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

sys.path.insert(0, os.path.abspath(".."))
from src.data import build_data_loader, raw_mnist
from src.model import build_resnet18
from src.settings import Settings, ray_init_with_repo

This notebook runs inside an Anyscale workspace, so it uses `ray_init_with_repo()` instead of a bare `ray.init()`. A workspace notebook has no job-level runtime environment, so without it a Ray Train worker that lands on a different process, or a different node, than this kernel fails to import `src.xxx` at all. `ray_init_with_repo()` ships this repo to every worker as a `runtime_env` working directory, and falls back to a bare `ray.init()` when a runtime env is already provided (for example inside an Anyscale Job).

In [ ]:
settings = Settings.from_env()
ray_init_with_repo()
settings.describe()

## 1. When to use Ray Train

Use Ray Train when you face one of the following challenges:

|Challenge|Detail|Solution|
|---|---|---|
|**Speed and scale**|Training a large model, or on a large dataset, on one GPU can take days|Ray Train distributes the same training loop across many GPUs and many nodes|
|**Cluster setup toil**|Provisioning workers, installing the right NCCL version, and writing hostfiles by hand doesn't scale past a laptop|Ray Train and the Anyscale autoscaler provision and configure the cluster for you|
|**Debugging distributed runs**|SSH-ing into N nodes to find one hung or crashed worker gets harder as N grows|The Ray dashboard shows worker state, logs, and GPU utilization for the whole run in one place|
|**Node failure**|A single pre-emption or hardware fault used to kill the entire job, losing all progress|Ray Train checkpoints automatically and restarts the worker group after a failure|
|**Not rewriting the loop**|Rewriting a training loop from scratch for a distributed framework is a real, ongoing cost|Ray Train wraps your existing PyTorch loop with a handful of utility calls; the loop's structure doesn't change|

|<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/multi_gpu_pytorch_v4.png" width="900px" loading="lazy">|
|:--|
|Schematic overview of DistributedDataParallel (DDP) training: (1) the model is replicated from the <code>GPU rank 0</code> to all other workers; (2) each worker receives a shard of the dataset and processes a mini-batch; (3) during the backward pass, gradients are averaged across GPUs; (4) checkpoint and metrics from rank 0 GPU are saved to the persistent storage.|

## 2. Update the training loop

Here is the full training function this notebook builds up to, section by section. It is the same function as `train_loop_per_worker` in `src/train_ray_train.py`, and the helper functions it calls (`load_checkpoint_state`, `report_checkpoint`, `maybe_fail_once`) are defined further down, the same order the script defines them in.

In [ ]:
def train_loop_per_worker(config: dict) -> None:
    ctx = ray.train.get_context()
    world_size, rank = ctx.get_world_size(), ctx.get_world_rank()
    print(f"[rank {rank}/{world_size}] node_rank={ctx.get_node_rank()} host={os.uname().nodename}")

    model = ray.train.torch.prepare_model(build_resnet18())  # device placement + DistributedDataParallel
    criterion = CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=config["lr"])

    per_worker_batch = config["global_batch_size"] // world_size
    data_loader = ray.train.torch.prepare_data_loader(  # DistributedSampler + batches moved to the device
        build_data_loader(config["data_root"], per_worker_batch, config["subset_size"])
    )

    start_epoch = load_checkpoint_state(model, optimizer)
    for epoch in range(start_epoch, config["num_epochs"]):
        maybe_fail_once(config, epoch)
        if world_size > 1:
            data_loader.sampler.set_epoch(epoch)
        loss = torch.tensor(0.0)
        for images, labels in data_loader:  # no .to(device): prepare_data_loader did it
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        metrics = {"loss": loss.item(), "epoch": epoch}
        if rank == 0:
            print(metrics)
        report_checkpoint(model, optimizer, metrics, epoch)

<div class="alert alert-block alert-info">

<b>Why keep <code>global_batch_size</code> fixed instead of the per-worker batch size?</b>

`per_worker_batch = global_batch_size // world_size` means adding workers shrinks each worker's batch, not the total amount of data processed per optimizer step. That keeps the gradient noise, and therefore the learning rate you already tuned, roughly the same as you scale from 2 workers to 8. If you instead kept the per-worker batch size fixed and let the global batch size grow with `world_size`, you would need to retune the learning rate every time you changed the worker count.

</div>

Notice what did **not** change from a single-GPU loop: no `images.to("cuda")`, no manual `DistributedSampler` wiring beyond `set_epoch`, and the forward and backward pass are exactly what you would write for one GPU. `sampler.set_epoch(epoch)` is guarded on `world_size > 1` because a single-worker `DataLoader` from `build_data_loader` has no distributed sampler to reshuffle.

### 2.1. Key concepts in Ray Train

Ray Train is built around [four key concepts](https://docs.ray.io/en/latest/train/overview.html):

1. **Training function** (`train_loop_per_worker` above): a Python function containing your model training logic.
2. **Training worker(s)**: processes that each run the training function, one shard of the data and one replica of the model per worker.
3. **ScalingConfig**: specifies the number of training workers and the compute resources (CPUs, GPUs) each one needs.
4. **Trainer**: a controller process that creates and monitors the workers according to the scaling configuration, and tracks training progress.

|<img src="https://docs.ray.io/en/latest/_images/overview.png" width="700px" loading="lazy">|
|:--|
|High-level architecture of how Ray Train coordinates the trainer, the workers, and storage.|

## 3. Migrate the model

The function that builds and instantiates the model stays exactly as it was for single-GPU training.

In [ ]:
build_resnet18

To make it distributed, [`ray.train.torch.prepare_model`](https://docs.ray.io/en/latest/train/api/doc/ray.train.torch.prepare_model.html) moves the model to the correct device and wraps it in PyTorch's `DistributedDataParallel`, replacing the `model.to("cuda")` call from notebook 01. `train_loop_per_worker` calls it as `model = ray.train.torch.prepare_model(build_resnet18())`.

<div class="alert alert-block alert-info">

<code>prepare_model()</code> takes a few more arguments worth knowing about:

<ul>
  <li><code>move_to_device</code>: whether to move the model to the correct device; set to <code>False</code> only if you want to move it yourself.</li>
  <li><code>parallel_strategy</code>: <code>"ddp"</code> (the default, used here) or <code>"fsdp"</code> to wrap the model in <code>FullyShardedDataParallel</code> instead, for models too large to replicate whole on every GPU.</li>
  <li><code>parallel_strategy_kwargs</code>: extra arguments passed through to the DDP or FSDP wrapper.</li>
</ul>

</div>

## 4. Migrate the dataset

[`ray.train.torch.prepare_data_loader`](https://docs.ray.io/en/latest/train/api/doc/ray.train.torch.prepare_data_loader.html) prepares a PyTorch `DataLoader` for distributed training. `train_loop_per_worker` calls it as `ray.train.torch.prepare_data_loader(build_data_loader(config["data_root"], per_worker_batch, config["subset_size"]))`, where `build_data_loader` is the unchanged function from `src/data.py`.

`prepare_data_loader` does two things:

* **Adds a `DistributedSampler`** so each worker sees a distinct shard of the dataset instead of the whole thing.
* **Wraps the `DataLoader`** to move batches to the correct device automatically, using a separate CUDA stream so data transfer can overlap with computation. That is the device transfer that replaces `images.to("cuda")`.

<div class="alert alert-block alert-info">

<b>Data too large to download on every node?</b>

Every worker here downloads MNIST independently, guarded by a file lock, which is fine for a small dataset. If your dataset is too large to duplicate on every node, Appendix A1 covers loading it with Ray Data and reading a per-worker shard with <code>get_dataset_shard()</code> instead.

</div>

## 5. Report metrics and checkpoints

`report_checkpoint` is what turns local training progress into something Ray Train (and later, Ray Tune) can see, resume, and use for fault tolerance.

In [ ]:
def report_checkpoint(model, optimizer, metrics: dict, epoch: int) -> None:
    """Every rank calls ray.train.report (it is a barrier). Only rank 0 attaches a checkpoint."""
    with tempfile.TemporaryDirectory() as tmp:
        checkpoint = None
        if ray.train.get_context().get_world_rank() == 0:
            torch.save(model.module.state_dict(), os.path.join(tmp, "model.pt"))  # .module unwraps DDP
            torch.save(optimizer.state_dict(), os.path.join(tmp, "optimizer.pt"))
            torch.save({"epoch": epoch}, os.path.join(tmp, "extra_state.pt"))
            checkpoint = Checkpoint.from_directory(tmp)
        ray.train.report(metrics, checkpoint=checkpoint)

<div class="alert alert-block alert-warning">

Only rank 0 builds a checkpoint (every rank has the same weights under DDP, so saving from every rank would be wasteful), but <b>every</b> rank still calls <code>ray.train.report</code>. It is a barrier: Ray Train blocks until every worker has reported before moving on, whether or not that worker attached a checkpoint.

</div>

The metrics dict printed on rank 0 in `train_loop_per_worker` (`{"loss": ..., "epoch": ...}`) is the same dict passed to `ray.train.report`, so what you see printed is exactly what ends up in `result.metrics` and `result.metrics_dataframe` later in this notebook.

|<img src="https://docs.ray.io/en/latest/_images/checkpoint_lifecycle.png" width="800">|
|:--|
|The checkpoint lifecycle, from a local temporary directory to persistent storage.|

To resume training, `train_loop_per_worker` first checks for an existing checkpoint with `ray.train.get_checkpoint()`. `load_checkpoint_state` restores the model, the optimizer, and the epoch to resume from, returning `0` when there is nothing to restore:

In [ ]:
def load_checkpoint_state(model, optimizer) -> int:
    """Restore model, optimizer, and epoch from the latest reported checkpoint.

    Returns the epoch to start from (0 when there is nothing to restore).
    """
    checkpoint = ray.train.get_checkpoint()
    if not checkpoint:
        return 0
    with checkpoint.as_directory() as ckpt_dir:
        model.module.load_state_dict(torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu"))
        optimizer.load_state_dict(torch.load(os.path.join(ckpt_dir, "optimizer.pt"), map_location="cpu"))
        start_epoch = torch.load(os.path.join(ckpt_dir, "extra_state.pt"))["epoch"] + 1
    print(f"[rank {ray.train.get_context().get_world_rank()}] resuming from epoch {start_epoch}")
    return start_epoch

Finally, `maybe_fail_once` is a teaching aid, not something you would ship: it crashes the last-ranked worker exactly once, at epoch 1, so section 12 can demonstrate automatic recovery on a real cluster instead of only describing it.

In [ ]:
def maybe_fail_once(config: dict, epoch: int) -> None:
    """Teaching aid for this notebook: crash the last worker once at epoch 1 so FailureConfig recovers."""
    if not config.get("fail_once") or epoch != 1:
        return
    ctx = ray.train.get_context()
    if ctx.get_world_rank() != ctx.get_world_size() - 1:
        return
    marker = os.path.join(config["fail_marker_dir"], "already_failed")
    if os.path.exists(marker):
        return
    os.makedirs(config["fail_marker_dir"], exist_ok=True)
    open(marker, "w").close()
    raise RuntimeError("Simulated worker failure, on purpose, to show automatic recovery")

With these three helpers defined, `train_loop_per_worker` from section 2 is now fully runnable. See the [checkpoints](https://docs.ray.io/en/latest/train/user-guides/checkpoints.html) guide for more on saving and loading patterns.

## 6. Set the runtime configuration (RunConfig)

`RunConfig` controls where a run's results live and how it behaves under failure.

In [ ]:
example_run_config = RunConfig(
    name="example-run-name",
    storage_path=settings.storage_path,
    checkpoint_config=CheckpointConfig(num_to_keep=2),
    failure_config=FailureConfig(max_failures=2),
)
example_run_config

* `name` and `storage_path` together identify a run's directory on persistent storage. **Reusing the same `name` and `storage_path` on a later call is what resumes a run** in Ray Train v2, covered in section 12.
* `storage_path` must be reachable from every node in the cluster, not just the head: a worker on any node needs to write its checkpoint there, and a later run (possibly on different nodes, after the cluster has scaled up or down) needs to read it back. See the [persistent storage](https://docs.ray.io/en/latest/train/user-guides/persistent-storage.html) guide.
* `CheckpointConfig(num_to_keep=2)` keeps only the two most recent checkpoints on storage, so a long run does not accumulate one checkpoint per epoch forever.
* `FailureConfig(max_failures=2)` is the automatic-retry mechanism covered in section 12: Ray Train restarts the worker group, up to two times, instead of failing the whole job on the first worker crash.

## 7. Configure scale and GPUs (ScalingConfig)

In [ ]:
example_scaling_config = ScalingConfig(num_workers=settings.num_workers, use_gpu=settings.use_gpu)
example_scaling_config

<div class="alert alert-block alert-info">

<a href="https://docs.ray.io/en/latest/train/api/doc/ray.train.ScalingConfig.html" target="_blank">ScalingConfig</a> has a few more knobs worth knowing:

<ul>
  <li><code>resources_per_worker</code>: CPU, GPU, or custom resources each worker needs; useful when a worker needs more than one GPU, or extra CPUs for data loading.</li>
  <li><code>accelerator_type</code>: pins workers to a specific accelerator (for example an A100), instead of accepting any GPU the autoscaler provisions.</li>
  <li><code>num_workers=(min, max)</code>: elastic training, where Ray Train adjusts the worker count within a range instead of a fixed number.</li>
</ul>

Before raising <code>num_workers</code>, check it against your workspace's GPU cap: the activity in this notebook uses four workers, which is this repo's compute config sized for exactly that (see section 13).

</div>

## 8. Launch the distributed training job

<div class="alert alert-block alert-warning">

<b>The worker group startup timeout defaults to 60 seconds.</b> That is tuned for a cluster where the GPU workers already exist, not for one that scales up from zero, which is the default everywhere in this repo. Provisioning the first GPU node of a run can take three to six minutes on its own. Without raising the timeout, a run against a cold cluster dies almost immediately with <code>WorkerGroupStartupTimeoutError</code>, not because anything is broken, but because the workers genuinely have not had time to start yet.

</div>

In [ ]:
os.environ.setdefault("RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S", "1800")

Here is the same distributed-data-parallel diagram from section 1, annotated with how Ray Train achieves it:

|<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/multi_gpu_pytorch_annotated_v5.png" width="70%" loading="lazy">|
|:--|

`build_trainer` bundles the `ScalingConfig` and `RunConfig` from sections 6 and 7 with the training function into a [`TorchTrainer`](https://docs.ray.io/en/latest/train/api/doc/ray.train.torch.TorchTrainer.html). It takes a `run_name` explicitly (rather than hardcoding one) so later sections can build fresh trainers, for the failure demo and the activity, without repeating this setup.

In [ ]:
def build_trainer(settings: Settings, run_name: str, fail_once: bool = False) -> TorchTrainer:
    train_loop_config = settings.as_train_loop_config()
    if fail_once:
        train_loop_config.update(
            fail_once=True, fail_marker_dir=os.path.join(settings.storage_path, "markers", run_name)
        )
    return TorchTrainer(
        train_loop_per_worker,
        train_loop_config=train_loop_config,
        scaling_config=ScalingConfig(num_workers=settings.num_workers, use_gpu=settings.use_gpu),
        run_config=RunConfig(
            name=run_name,
            storage_path=settings.storage_path,
            checkpoint_config=CheckpointConfig(num_to_keep=2),
            failure_config=FailureConfig(max_failures=2),
        ),
    )


def default_run_name() -> str:
    import datetime

    return "mnist-resnet18-" + datetime.datetime.now(datetime.UTC).strftime("%Y%m%d-%H%M%S")

Calling `.fit()` starts the run and blocks until it completes.

<div class="alert alert-block alert-info">

Watch the output as this runs: every worker prints its own <code>host=</code> value from <code>train_loop_per_worker</code>. Two distinct hosts means this run is multi-node, with no hostfile written by hand anywhere. The Ray Train tab in the Anyscale dashboard shows these same workers live, including which node each one landed on.

</div>

In [ ]:
run_name = default_run_name()
print(f"run_name={run_name}  (export RUN_NAME={run_name} to resume this run later)")
result = build_trainer(settings, run_name).fit()

## 9. Access the training results

In [ ]:
result

In [ ]:
print("final metrics:", result.metrics)
print("checkpoint:", result.checkpoint)
print("run directory:", result.path)

In [ ]:
!ls {result.path}

In [ ]:
result.metrics_dataframe

See the [results](https://docs.ray.io/en/latest/train/user-guides/results.html) guide for the full `Result` API.

## 10. Use the checkpointed model

In [ ]:
ckpt = result.checkpoint
with ckpt.as_directory() as ckpt_dir:
    state_dict = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu")
    loaded_model = build_resnet18()
    loaded_model.load_state_dict(state_dict)
    loaded_model.eval()

Generate predictions on nine random images from the MNIST training set. This runs on the head node's CPU, which is exactly why the checkpoint was saved without device state baked in.

In [ ]:
from src.data import MNIST_TRANSFORM

dataset = raw_mnist(settings.data_root, train=True)
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3

for i in range(1, cols * rows + 1):
    sample_idx = np.random.randint(0, len(dataset.data))
    img, label = dataset[sample_idx]
    normalized_img = MNIST_TRANSFORM(img)

    with torch.no_grad():
        prediction = loaded_model(normalized_img.unsqueeze(0)).argmax().item()

    figure.add_subplot(rows, cols, i)
    plt.title(f"label: {label}; pred: {prediction}")
    plt.axis("off")
    plt.imshow(img, cmap="gray")

## 11. See the workers across nodes

In [ ]:
for node in ray.nodes():
    resources = node["Resources"]
    print(
        f"alive={node['Alive']!s:5} CPU={resources.get('CPU', 0):.0f} "
        f"GPU={resources.get('GPU', 0):.0f} node_ip={node['NodeManagerAddress']}"
    )

The head node shows `GPU=0` by design: it runs the driver, the dashboard, and cluster bookkeeping, and never runs a training worker. Every GPU counted above belongs to a worker node that the autoscaler brought up for this run.

## 12. Recover from a failure, and resume

Ray Train gives you two mechanisms for handling failures, and they solve different problems:

* **Automatic retries.** `FailureConfig(max_failures=2)` from section 6 tells Ray Train to restart the worker group, up to twice, when a worker crashes mid-run. This is what keeps a multi-hour job alive through a transient node failure or pre-emption, with no action from you.
* **Resuming a finished-or-abandoned run later.** Ray Train v2 has no `Trainer.restore()`. Instead, you rerun with the **same** `RunConfig(name, storage_path)`, and `load_checkpoint_state` reads `ray.train.get_checkpoint()` inside the training function to pick up where the last checkpoint left off.

### 12.1. Automatic retries in action

This builds a new trainer with `fail_once=True`, which arms `maybe_fail_once` from section 5. Watch the output: epoch 0 completes and checkpoints normally, then the last-ranked worker deliberately raises at epoch 1. Ray Train detects the failure, restarts the entire worker group, and every rank prints `resuming from epoch 1` as `load_checkpoint_state` picks the run back up from the checkpoint saved after epoch 0.

In [ ]:
fail_run_name = "fault-tolerant-" + default_run_name()
fail_result = build_trainer(settings, fail_run_name, fail_once=True).fit()
print("final metrics after recovery:", fail_result.metrics)

You should see output shaped like this (host values and exact loss numbers will differ):

```text
[rank 0/2] node_rank=0 host=ip-10-0-1-23
[rank 1/2] node_rank=1 host=ip-10-0-3-45
{'loss': 1.83, 'epoch': 0}
RuntimeError: Simulated worker failure, on purpose, to show automatic recovery
... Ray Train restarts the worker group ...
[rank 0] resuming from epoch 1
[rank 1] resuming from epoch 1
{'loss': 0.94, 'epoch': 1}
final metrics after recovery: {'loss': 0.94, 'epoch': 1}
```

### 12.2. Resuming with the same RunConfig, no restore API

To resume, rerun with the identical `run_name` and `storage_path`, but one extra epoch. There is no `.restore()` call: `load_checkpoint_state` inside `train_loop_per_worker` is what makes this a resume instead of a fresh run, by finding the checkpoint already on storage from `fail_run_name` and picking up at `epoch=2`.

In [ ]:
import dataclasses

settings_plus_one_epoch = dataclasses.replace(settings, num_epochs=settings.num_epochs + 1)
resumed_result = build_trainer(settings_plus_one_epoch, fail_run_name).fit()
print("final metrics after resume:", resumed_result.metrics)

<div class="alert alert-block alert-warning">

Reusing <code>fail_run_name</code> here is what makes this a resume rather than a fresh run. Run names must be <b>unique per job</b> you don't intend to resume, and <b>stable</b> for a job you might need to resume later. There is no separate "resume" flag to set: the name and the storage path together are the whole mechanism.

</div>

## 13. Activity: run with four workers

<div class="alert alert-block alert-info">

1. Update the scaling configuration to use 4 GPU workers instead of `settings.num_workers`.
2. Give this run a **new** name. Reusing `run_name` from section 8 would resume that completed run instead of starting fresh with four workers, per the warning in section 12.
3. Run the trainer with the same hyperparameters, watching the Cluster tab in the Anyscale dashboard as the autoscaler brings up the extra GPU workers.
4. Compare `metrics_dataframe` against the two-worker run from section 9.

Use this snippet to guide you:

```python
# Hint: a fresh ScalingConfig, a fresh run name, same train_loop_config
four_worker_scaling_config = ScalingConfig(num_workers=..., use_gpu=settings.use_gpu)
four_worker_run_name = ...

trainer = TorchTrainer(
    train_loop_per_worker,
    train_loop_config=settings.as_train_loop_config(),
    scaling_config=...,
    run_config=RunConfig(name=..., storage_path=settings.storage_path),
)
four_worker_result = trainer.fit()
four_worker_result.metrics_dataframe
```

</div>

In [ ]:
# Write your solution here


<div class="alert alert-block alert-info">

<details>

<summary> Click here to see the solution </summary>

```python
four_worker_scaling_config = ScalingConfig(num_workers=4, use_gpu=settings.use_gpu)
four_worker_run_name = "mnist-resnet18-4workers-" + default_run_name()

trainer = TorchTrainer(
    train_loop_per_worker,
    train_loop_config=settings.as_train_loop_config(),
    scaling_config=four_worker_scaling_config,
    run_config=RunConfig(name=four_worker_run_name, storage_path=settings.storage_path),
)
four_worker_result = trainer.fit()
four_worker_result.metrics_dataframe
```

With `global_batch_size` unchanged, four workers means half the per-worker batch size of the two-worker run, and each epoch should complete faster since more GPUs are processing shards in parallel. The final loss should land close to the two-worker run, since `global_batch_size` (and therefore the learning dynamics) did not change, only the degree of parallelism did.

</details>

</div>

## 14. What changed from v1, and Ray Train in production

<div class="alert alert-block alert-info">

<b>What changed from v1</b>

If you have seen older Ray Train code, or an older blog post, the biggest change is how resuming works: v1's <code>Trainer.restore(path)</code> is gone. See <code>docs/ray-train-v1-to-v2.md</code> in this repo for the full v1-to-v2 map, including how <code>ray.tune.Tuner(trainer)</code> became the driver-function pattern notebook 03 covers next.

</div>

Ray Train is used in production well beyond MNIST-sized jobs:

1. Canva uses Ray Train and Ray Data to cut Stable Diffusion pretraining costs by 3.7x. Read the [Anyscale blog post](https://www.anyscale.com/blog/scalable-and-cost-efficient-stable-diffusion-pre-training-with-ray).
2. Anyscale's managed [Ray Train runtime](https://docs.anyscale.com/runtime/train) builds on the fault tolerance and checkpointing mechanics from this notebook, for workloads that run for days rather than minutes.

Next: notebook 03 sweeps the hyperparameters of this same training loop with Ray Tune.

## Further reading

| Topic | Link |
|---|---|
| Ray Train overview | https://docs.ray.io/en/latest/train/overview.html |
| PyTorch getting started | https://docs.ray.io/en/latest/train/getting-started-pytorch.html |
| `prepare_model` | https://docs.ray.io/en/latest/train/api/doc/ray.train.torch.prepare_model.html |
| `prepare_data_loader` | https://docs.ray.io/en/latest/train/api/doc/ray.train.torch.prepare_data_loader.html |
| `ray.train.report` | https://docs.ray.io/en/latest/train/api/doc/ray.train.report.html |
| Checkpoints | https://docs.ray.io/en/latest/train/user-guides/checkpoints.html |
| Persistent storage | https://docs.ray.io/en/latest/train/user-guides/persistent-storage.html |
| Results | https://docs.ray.io/en/latest/train/user-guides/results.html |
| Fault tolerance | https://docs.ray.io/en/latest/train/user-guides/fault-tolerance.html |
| Accelerators | https://docs.ray.io/en/latest/train/user-guides/using-accelerators.html |
| Monitoring | https://docs.ray.io/en/latest/train/user-guides/monitoring-logging.html |
| The v2 migration issue | https://github.com/ray-project/ray/issues/49454 |
| Ray Train on Anyscale | https://docs.anyscale.com/runtime/train |
| Template: distributing-pytorch | https://github.com/anyscale/templates/tree/main/templates/distributing-pytorch |
| Template: ray_train_workloads | https://github.com/anyscale/templates/tree/main/templates/ray_train_workloads |
| Ray Train specialization course | https://anyscale.coursifai.com/learning-paths/ray-train-specialization |